#  **Identificando discurso de ódio pelo Qwen 3**

**Autores:** Luiz Davi da Silva Aguiar e Adonias Caetano de Oliveira

## **Installation**

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
!pip install Unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 14.7 MB/s eta 0:00:00


## **Dataset**

In [ ]:
import pandas as pd
import re
import random
import numpy as np
import torch

In [ ]:
url = 'https://drive.google.com/file/d/1ur5voOHqK7JasJF6uinmQXXvlAxHJ8ni/view?usp=sharing'
file_id = url.split('/')[-2]
read_url='https://drive.google.com/uc?id=' + file_id

dataset = pd.read_csv(read_url, usecols=['text', 'hatespeech_comb'])

dataset.head()

,text,hatespeech_comb
0,Estou trabalhando num sistema onde haverá comp...,0
1,@SadBoyKevyn \nNunca penses que algo 'é o mais...,0
2,Começa pelo seu namorado @badgalerica https://...,0
3,"o discurso do socialismo pode até ser bonito, ...",0
4,RT @mayaramellon: é cheio de opinião mas não t...,0


## **Unsloth**

In [ ]:
from unsloth import FastLanguageModel


fourbit_models = [
    "unsloth/Qwen3-1.7B-unsloth-bnb-4bit", # Qwen 14B 2x faster
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    "unsloth/Qwen3-14B-unsloth-bnb-4bit",
    "unsloth/Qwen3-32B-unsloth-bnb-4bit",
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/Phi-4",
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/orpheus-3b-0.1-ft-unsloth-bnb-4bit" # [NEW] We support TTS models!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    max_seq_length = 2048,   # Context length - can be longer, but uses more memory
    load_in_4bit = True,     # 4bit uses much less memory
    load_in_8bit = False,    # A bit more accurate, uses 2x memory
    full_finetuning = False, # We have full finetuning now!
    # token = "hf_...",      # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find `steps_per_generation` in grpo_trainer
Unsloth: Could not find `generation_batch_size` in grpo_trainer
==((====))==  Unsloth 2026.8.12: Fast Qwen3 patching. Transformers: 5.13.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,           # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,  # Best to choose alpha = rank or rank*2
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,   # We support rank stabilized LoRA
    loftq_config = None,  # And LoftQ
)

Unsloth 2026.8.12 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Inference"></a>
### **Inference**
De acordo com a equipe `Qwen-3`, as configurações recomendadas para inferência de raciocínio são `temperature = 0.6, top_p = 0.95, top_k = 20`

Para inferência baseada em bate-papo normal, `temperature = 0.7, top_p = 0.8, top_k = 20`

In [ ]:
respostas_llm = {}

sentencas = dataset["text"].to_list()
targets = dataset["hatespeech_comb"].to_list()

respostas_llm["text"] = sentencas
respostas_llm["target"] = targets
respostas_llm['predicted'] = []

for sentenca in sentencas:

  prompt = f"""
    Você é um moderador de conteúdo. Classifique a frase assim:

    - "1" se a frase contém discurso de ódio (hate speech)
    - "0" se a frase não contém nenhum tipo de discurso de ódio (non-hate speech)


    Regras:
    - Sua saída deve ser apenas um número inteiro (0 ou 1)
    - Não inclua nenhum texto adicional na resposta

    Frase: "{sentenca}"
    """

  messages = [
      {"role" : "user", "content" :  prompt}
  ]
  text = tokenizer.apply_chat_template(
      messages,
      tokenize = False,
      add_generation_prompt = True, # Must add for generation
      enable_thinking = True, # Disable thinking
  )

  FastLanguageModel.for_inference(model) # Enable native 2x faster inference

  inputs = tokenizer(text, return_tensors = "pt").to("cuda")
  output = model.generate(
    **inputs,
    max_new_tokens=1024,
    temperature=0.0,
    top_p=0.95,
    top_k=20
)
  resposta_ids = output[0]
  resposta_texto = tokenizer.decode(resposta_ids, skip_special_tokens=True)


  #respostas_llm['predicted'].append(resposta_texto.split("</think>")[1].strip())
  try:
    # Attempt to extract the number after "texto." (for chat template where the model adds "texto." and then the number)
    predicted_value = int(resposta_texto.split("texto.")[1].strip())
  except (ValueError, IndexError):
    # Fallback to general extraction or default if "texto." is not present or conversion fails
    # This attempts to find the last integer in the string as a fallback
    nums = re.findall(r'\b\d+\b', resposta_texto)
    if nums:
        predicted_value = int(nums[-1])
    else:
        predicted_value = -1 # Or any other indicator for failure

  respostas_llm['predicted'].append(predicted_value)


Both `max_new_tokens` (=1024) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

In [ ]:

from google.colab import files

df = pd.DataFrame(respostas_llm)
df.to_excel('respostas_qwen3.xlsx', index=False)

files.download('respostas_qwen3.xlsx')


In [ ]:
from sklearn.metrics import confusion_matrix

y_true = df["target"]
y_pred = df["predicted"]

cm = confusion_matrix(y_true, y_pred)
print(cm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Non-Hate", "Hate"])
disp.plot()

plt.title("Matriz de Confusão - Qwen3")
plt.show()

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

y_true = df["target"]
y_pred = df["predicted"]

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"F1-score: {f1:.4f}")

In [ ]:
print(classification_report(y_true, y_pred, target_names=["Non-Hate", "Hate"]))